In [ ]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Generate unique identifiers for this ETL run
SSIDGUID = str(uuid.uuid4())
ETLRunTime = datetime.now()
IncrementalExtractTime = datetime.now()

# Configuration variables matching the original stored procedure
MasterSourceSystemName = ('DJJStarSchema')
SourceSubSystemName = ('')
SourceDatabaseName = ('DJJODS')
SourceTableName = ('DJJODS.dbo.ODS_NUE_Forecasts')
DestinationSchemaName = ('NUE')
DestinationTableName = ('factConsumptionForecast')
DestinationDatabaseName = ('DJJStarSchema')
ETLType = ('Incremental')
BatchSize = 1000
PkgName = ('NUE.etl_factConsumptionForecast')
ETLTemplate = ('')
PkgGUID = str(uuid.uuid4())
DJJLastUpdateTime = datetime.now()

# Catalog mappings
# source_catalog: bronze layer for operational source systems (DJJODS -> bronze_non_prod)
source_catalog = "{bronze_non_prod}"
# target_catalog: temporary / enriched target layer (djj_enriched_non_prod)
target_catalog = "{djj_enriched_non_prod}"
# federated_starschema_catalog: star schema / federated analytical layer (djj_star_schema)
federated_starschema_catalog = "{djj_star_schema}"

# Audit counters
Success = '0'
RowsInserted = 0
RowsUpdated = 0
RowsDeleted = 0
ErrorRowCount = 0
TableInitialRowCount = 0
TableFinalRowCount = 0
ETLStatus = ('')
ExtractRowCount = 0
SSISGUID = SSIDGUID

### Execute the Initial Metadata Logging

In [ ]:
%run /Workspace/Shared/metadata_framework/metadata_logger

In [ ]:
# Insert/Update records in ETL Master
logger = ETLLogger(spark)

ETLID = logger.log_etl(
    MasterSourceSystemName=MasterSourceSystemName,
    SourceSubSystemName=SourceSubSystemName,
    SourceDatabaseName=SourceDatabaseName,
    SourceTableName=SourceTableName,
    DestinationDatabaseName=DestinationDatabaseName,
    DestinationTableName=DestinationTableName,
    PkgName=PkgName,
    PkgGUID=PkgGUID,
    ETLTemplate=ETLTemplate,
    ETLType=ETLType,
    CheckpointsEnabled=True
)

print(ETLID)

In [ ]:
# Insert initial record in ETLExecutionLog table
exec_logger = ETLExecutionLogger(spark)
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success=Success,
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [ ]:
# Retrieve ETLExecutionID from the execution log for use in downstream inserts
ETLExecutionID = spark.sql(f"""
    SELECT ETLExecutionID
    FROM {target_catalog}.metadata.etlexecutionlog
    WHERE lower(trim(CAST(ETLID AS STRING))) = lower(trim('{ETLID}'))
      AND lower(trim(SSISGUID)) = lower(trim('{SSIDGUID}'))
""").first()[0]

### ETL Logic Starts Here

In [ ]:
# Get initial row count of the target fact table
# NUE schema in SQL -> NUE schema in Databricks (star schema)
TableInitialRowCount = spark.sql(f"""
    SELECT COUNT(1) AS cnt
    FROM {target_catalog}.NUE.factConsumptionForecast
""").first()["cnt"]
print(f"Initial row count: {TableInitialRowCount}")

# Get TargetLastUpdatedDate from ODS_NUE_Forecasts source (bronze) table
# DJJLastUpdateTime in source bronze tables is mapped to BronzeTimestampUTC per prompt rules
target_last_updated_row = spark.sql(f"""
    SELECT COALESCE(
        CAST(MAX(CAST(BronzeTimestampUTC AS DATE)) AS TIMESTAMP),
        CAST('1900-01-01' AS TIMESTAMP)
    ) AS TargetLastUpdatedDate
    FROM {source_catalog}.djj.ODS_NUE_Forecasts
""").first()
TargetLastUpdatedDate = str(target_last_updated_row["TargetLastUpdatedDate"])
print(f"TargetLastUpdatedDate: {TargetLastUpdatedDate}")

### Source Query – Create Temp Table #CVPForecastedConsumption

In [ ]:
# Key conversion notes for this temp table:
#
# Date Key formula: CONVERT(INT, CONVERT(VARCHAR(12), DATEADD(MONTH, N, DATEFROMPARTS(Year,Month,1)), 112))
#   Produces YYYYMMDD integer date keys. Spark equivalent:
#   CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(Year, Month, 1), N), 'yyyyMMdd') AS INT)
#
# Schema mappings:
#   DJJODS.dbo.*                              -> source_catalog.djj.*
#   DJJStarSchema.Brk.dimEnterpriseGradeGroup -> federated_starschema_catalog.djj_brk.dimEnterpriseGradeGroup
#   DJJStarSchema.Nue.dimNucorMillLocations   -> federated_starschema_catalog.NUE.dimNucorMillLocations
#   DJJStarSchema.Brk.dimConsumer             -> federated_starschema_catalog.djj_brk.dimConsumer
#
# DJJDeletedFlag -> IsDeleted per prompt rules
# ISNULL(x, y)   -> COALESCE(x, y)
#
# The CASE WHEN used in the LEFT JOIN ON clause for the EnterpriseGradeGroup lookup:
#   CASE WHEN rmfal.RawMaterialName = '#1 Busheling & Bundles' THEN 'Busheling' ELSE rmfal.RawMaterialName END
#   lower(trim()) applied on both sides of the JOIN ON condition.
#   Per rules: lower(trim()) on CASE WHEN -> applied inside the WHEN condition only, not THEN.
#
# ALTER TABLE ADD ConsumerKey + UPDATE via LEFT JOIN chain resolved inline in the CREATE TABLE AS SELECT.
# ALTER TABLE ADD DJJGeneratedFlag, ETLSSExecutionID, DJJLastUpdateTime + UPDATE resolved inline.
# DJJLastUpdateTime written to enriched/temp target layer -> EnrichedTimestampUTC per prompt rules.
#
# PRIMARY KEY CLUSTERED constraint on temp table -> omitted (not supported in Delta Lake).
# UNION across all 7 forecast period blocks preserves deduplication as in original.
#
# Static IN lists (e.g., IN ('UsageGT','UsagePercent') and IN ('Cuts','Home',...)) kept as-is per Rule 20.
# lower(trim()) applied on both sides of those literal string comparisons.

In [ ]:
# Create temp table equivalent for #CVPForecastedConsumption
# Combines 7 forecast periods (PriorPeriodActual + Forecast30 through Forecast180)
# using UNION for deduplication across all periods, matching the original SQL.
#
# ConsumerKey resolved inline via LEFT JOIN chain:
#   NucorOrgID -> NUE.dimNucorMillLocations (NucorOrgID) -> djj_brk.dimConsumer (ConsumerID via DJJConsumerID)
# This replaces the original ALTER TABLE ADD ConsumerKey + UPDATE pattern (unsupported in Delta).
#
# DJJGeneratedFlag, ETLSSExecutionID, EnrichedTimestampUTC resolved inline.
# This replaces the original ALTER TABLE ADD + UPDATE pattern for those columns.

spark.sql(f"DROP TABLE IF EXISTS {target_catalog}.temp.CVPForecastedConsumption")

qs_create_temp = f"""
    CREATE TABLE {target_catalog}.temp.CVPForecastedConsumption
    USING DELTA
    AS
    SELECT
        base.ConsumptionForecastDateKey,
        base.NucorOrgID,
        base.EnterpriseGradeGroupKey,
        base.ForecastPeriod,
        base.ConsumptionForecastPeriodDateKey,
        base.ForecastWgt,
        base.ForecastPct,
        COALESCE(cons.ConsumerKey, -1)  AS ConsumerKey,
        0                               AS DJJGeneratedFlag,
        {ETLExecutionID}                AS ETLSSExecutionID,
        current_timestamp()             AS EnrichedTimestampUTC
    FROM (

        -- Prior Period Actual Wgts and Pcts (ForecastPeriod = 0, ConsumptionForecastPeriodDateKey offset = +-1 month(s))
        SELECT
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)  AS ConsumptionForecastDateKey,
            mm.Code                                                                                AS NucorOrgID,
            COALESCE(b.EnterpriseGradeGroupKey, -1)                                               AS EnterpriseGradeGroupKey,
            0                                                                          AS ForecastPeriod,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), -1), 'yyyyMMdd') AS INT)  AS ConsumptionForecastPeriodDateKey,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsageGT'))      THEN COALESCE(PriorPeriodActual, 0) * 2240.0 ELSE 0 END) AS ForecastWgt,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsagePercent')) THEN COALESCE(PriorPeriodActual, 0)         ELSE 0 END) AS ForecastPct
        FROM {source_catalog}.djj.ODS_NUE_Forecasts ff
            INNER JOIN {source_catalog}.djj.ODS_MDS_NUEMillNames mm
                ON lower(trim(CAST(ff.DivisionId AS STRING))) = lower(trim(CAST(mm.Code AS STRING)))
               AND mm.IsDeleted = 0
            INNER JOIN {source_catalog}.djj.ODS_NUE_RawMaterialForecastActualLines rmfal
                ON lower(trim(CAST(ff.Id AS STRING))) = lower(trim(CAST(rmfal.ForecastId AS STRING)))
               AND rmfal.IsDeleted = 0
            LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimEnterpriseGradeGroup b
                ON lower(trim(
                    CASE WHEN lower(trim(rmfal.RawMaterialName)) = lower(trim('#1 Busheling & Bundles'))
                         THEN 'Busheling'
                         ELSE rmfal.RawMaterialName END
                   )) = lower(trim(b.EnterpriseGradeGroup))
        WHERE ff.IsDeleted = 0
          AND lower(trim(rmfal.Discriminator)) IN (lower(trim('UsageGT')), lower(trim('UsagePercent')))
          AND lower(trim(rmfal.ParentGradeGroupName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.RawMaterialName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.ParentGradeGroupName)) IN (
              lower(trim('Cuts')), lower(trim('Home')), lower(trim('Primes')),
              lower(trim('Secondaries')), lower(trim('Shred')), lower(trim('Subs'))
          )
        GROUP BY
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT),
            mm.Code,
            b.EnterpriseGradeGroupKey,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), -1), 'yyyyMMdd') AS INT)

        UNION

        -- Forecast 30 (ForecastPeriod = 1, ConsumptionForecastPeriodDateKey offset = +0 month(s))
        SELECT
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)  AS ConsumptionForecastDateKey,
            mm.Code                                                                                AS NucorOrgID,
            COALESCE(b.EnterpriseGradeGroupKey, -1)                                               AS EnterpriseGradeGroupKey,
            1                                                                          AS ForecastPeriod,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)  AS ConsumptionForecastPeriodDateKey,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsageGT'))      THEN COALESCE(Forecast30, 0) * 2240.0 ELSE 0 END) AS ForecastWgt,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsagePercent')) THEN COALESCE(Forecast30, 0)         ELSE 0 END) AS ForecastPct
        FROM {source_catalog}.djj.ODS_NUE_Forecasts ff
            INNER JOIN {source_catalog}.djj.ODS_MDS_NUEMillNames mm
                ON lower(trim(CAST(ff.DivisionId AS STRING))) = lower(trim(CAST(mm.Code AS STRING)))
               AND mm.IsDeleted = 0
            INNER JOIN {source_catalog}.djj.ODS_NUE_RawMaterialForecastActualLines rmfal
                ON lower(trim(CAST(ff.Id AS STRING))) = lower(trim(CAST(rmfal.ForecastId AS STRING)))
               AND rmfal.IsDeleted = 0
            LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimEnterpriseGradeGroup b
                ON lower(trim(
                    CASE WHEN lower(trim(rmfal.RawMaterialName)) = lower(trim('#1 Busheling & Bundles'))
                         THEN 'Busheling'
                         ELSE rmfal.RawMaterialName END
                   )) = lower(trim(b.EnterpriseGradeGroup))
        WHERE ff.IsDeleted = 0
          AND lower(trim(rmfal.Discriminator)) IN (lower(trim('UsageGT')), lower(trim('UsagePercent')))
          AND lower(trim(rmfal.ParentGradeGroupName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.RawMaterialName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.ParentGradeGroupName)) IN (
              lower(trim('Cuts')), lower(trim('Home')), lower(trim('Primes')),
              lower(trim('Secondaries')), lower(trim('Shred')), lower(trim('Subs'))
          )
        GROUP BY
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT),
            mm.Code,
            b.EnterpriseGradeGroupKey,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)

        UNION

        -- Forecast 60 (ForecastPeriod = 2, ConsumptionForecastPeriodDateKey offset = +1 month(s))
        SELECT
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)  AS ConsumptionForecastDateKey,
            mm.Code                                                                                AS NucorOrgID,
            COALESCE(b.EnterpriseGradeGroupKey, -1)                                               AS EnterpriseGradeGroupKey,
            2                                                                          AS ForecastPeriod,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 1), 'yyyyMMdd') AS INT)  AS ConsumptionForecastPeriodDateKey,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsageGT'))      THEN COALESCE(Forecast60, 0) * 2240.0 ELSE 0 END) AS ForecastWgt,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsagePercent')) THEN COALESCE(Forecast60, 0)         ELSE 0 END) AS ForecastPct
        FROM {source_catalog}.djj.ODS_NUE_Forecasts ff
            INNER JOIN {source_catalog}.djj.ODS_MDS_NUEMillNames mm
                ON lower(trim(CAST(ff.DivisionId AS STRING))) = lower(trim(CAST(mm.Code AS STRING)))
               AND mm.IsDeleted = 0
            INNER JOIN {source_catalog}.djj.ODS_NUE_RawMaterialForecastActualLines rmfal
                ON lower(trim(CAST(ff.Id AS STRING))) = lower(trim(CAST(rmfal.ForecastId AS STRING)))
               AND rmfal.IsDeleted = 0
            LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimEnterpriseGradeGroup b
                ON lower(trim(
                    CASE WHEN lower(trim(rmfal.RawMaterialName)) = lower(trim('#1 Busheling & Bundles'))
                         THEN 'Busheling'
                         ELSE rmfal.RawMaterialName END
                   )) = lower(trim(b.EnterpriseGradeGroup))
        WHERE ff.IsDeleted = 0
          AND lower(trim(rmfal.Discriminator)) IN (lower(trim('UsageGT')), lower(trim('UsagePercent')))
          AND lower(trim(rmfal.ParentGradeGroupName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.RawMaterialName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.ParentGradeGroupName)) IN (
              lower(trim('Cuts')), lower(trim('Home')), lower(trim('Primes')),
              lower(trim('Secondaries')), lower(trim('Shred')), lower(trim('Subs'))
          )
        GROUP BY
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT),
            mm.Code,
            b.EnterpriseGradeGroupKey,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 1), 'yyyyMMdd') AS INT)

        UNION

        -- Forecast 90 (ForecastPeriod = 3, ConsumptionForecastPeriodDateKey offset = +2 month(s))
        SELECT
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)  AS ConsumptionForecastDateKey,
            mm.Code                                                                                AS NucorOrgID,
            COALESCE(b.EnterpriseGradeGroupKey, -1)                                               AS EnterpriseGradeGroupKey,
            3                                                                          AS ForecastPeriod,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 2), 'yyyyMMdd') AS INT)  AS ConsumptionForecastPeriodDateKey,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsageGT'))      THEN COALESCE(Forecast90, 0) * 2240.0 ELSE 0 END) AS ForecastWgt,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsagePercent')) THEN COALESCE(Forecast90, 0)         ELSE 0 END) AS ForecastPct
        FROM {source_catalog}.djj.ODS_NUE_Forecasts ff
            INNER JOIN {source_catalog}.djj.ODS_MDS_NUEMillNames mm
                ON lower(trim(CAST(ff.DivisionId AS STRING))) = lower(trim(CAST(mm.Code AS STRING)))
               AND mm.IsDeleted = 0
            INNER JOIN {source_catalog}.djj.ODS_NUE_RawMaterialForecastActualLines rmfal
                ON lower(trim(CAST(ff.Id AS STRING))) = lower(trim(CAST(rmfal.ForecastId AS STRING)))
               AND rmfal.IsDeleted = 0
            LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimEnterpriseGradeGroup b
                ON lower(trim(
                    CASE WHEN lower(trim(rmfal.RawMaterialName)) = lower(trim('#1 Busheling & Bundles'))
                         THEN 'Busheling'
                         ELSE rmfal.RawMaterialName END
                   )) = lower(trim(b.EnterpriseGradeGroup))
        WHERE ff.IsDeleted = 0
          AND lower(trim(rmfal.Discriminator)) IN (lower(trim('UsageGT')), lower(trim('UsagePercent')))
          AND lower(trim(rmfal.ParentGradeGroupName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.RawMaterialName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.ParentGradeGroupName)) IN (
              lower(trim('Cuts')), lower(trim('Home')), lower(trim('Primes')),
              lower(trim('Secondaries')), lower(trim('Shred')), lower(trim('Subs'))
          )
        GROUP BY
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT),
            mm.Code,
            b.EnterpriseGradeGroupKey,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 2), 'yyyyMMdd') AS INT)

        UNION

        -- Forecast 120 (ForecastPeriod = 4, ConsumptionForecastPeriodDateKey offset = +3 month(s))
        SELECT
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)  AS ConsumptionForecastDateKey,
            mm.Code                                                                                AS NucorOrgID,
            COALESCE(b.EnterpriseGradeGroupKey, -1)                                               AS EnterpriseGradeGroupKey,
            4                                                                          AS ForecastPeriod,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 3), 'yyyyMMdd') AS INT)  AS ConsumptionForecastPeriodDateKey,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsageGT'))      THEN COALESCE(Forecast120, 0) * 2240.0 ELSE 0 END) AS ForecastWgt,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsagePercent')) THEN COALESCE(Forecast90, 0)         ELSE 0 END) AS ForecastPct
        FROM {source_catalog}.djj.ODS_NUE_Forecasts ff
            INNER JOIN {source_catalog}.djj.ODS_MDS_NUEMillNames mm
                ON lower(trim(CAST(ff.DivisionId AS STRING))) = lower(trim(CAST(mm.Code AS STRING)))
               AND mm.IsDeleted = 0
            INNER JOIN {source_catalog}.djj.ODS_NUE_RawMaterialForecastActualLines rmfal
                ON lower(trim(CAST(ff.Id AS STRING))) = lower(trim(CAST(rmfal.ForecastId AS STRING)))
               AND rmfal.IsDeleted = 0
            LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimEnterpriseGradeGroup b
                ON lower(trim(
                    CASE WHEN lower(trim(rmfal.RawMaterialName)) = lower(trim('#1 Busheling & Bundles'))
                         THEN 'Busheling'
                         ELSE rmfal.RawMaterialName END
                   )) = lower(trim(b.EnterpriseGradeGroup))
        WHERE ff.IsDeleted = 0
          AND lower(trim(rmfal.Discriminator)) IN (lower(trim('UsageGT')), lower(trim('UsagePercent')))
          AND lower(trim(rmfal.ParentGradeGroupName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.RawMaterialName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.ParentGradeGroupName)) IN (
              lower(trim('Cuts')), lower(trim('Home')), lower(trim('Primes')),
              lower(trim('Secondaries')), lower(trim('Shred')), lower(trim('Subs'))
          )
        GROUP BY
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT),
            mm.Code,
            b.EnterpriseGradeGroupKey,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 3), 'yyyyMMdd') AS INT)

        UNION

        -- Forecast 150 (ForecastPeriod = 5, ConsumptionForecastPeriodDateKey offset = +4 month(s))
        SELECT
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)  AS ConsumptionForecastDateKey,
            mm.Code                                                                                AS NucorOrgID,
            COALESCE(b.EnterpriseGradeGroupKey, -1)                                               AS EnterpriseGradeGroupKey,
            5                                                                          AS ForecastPeriod,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 4), 'yyyyMMdd') AS INT)  AS ConsumptionForecastPeriodDateKey,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsageGT'))      THEN COALESCE(Forecast150, 0) * 2240.0 ELSE 0 END) AS ForecastWgt,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsagePercent')) THEN COALESCE(Forecast90, 0)         ELSE 0 END) AS ForecastPct
        FROM {source_catalog}.djj.ODS_NUE_Forecasts ff
            INNER JOIN {source_catalog}.djj.ODS_MDS_NUEMillNames mm
                ON lower(trim(CAST(ff.DivisionId AS STRING))) = lower(trim(CAST(mm.Code AS STRING)))
               AND mm.IsDeleted = 0
            INNER JOIN {source_catalog}.djj.ODS_NUE_RawMaterialForecastActualLines rmfal
                ON lower(trim(CAST(ff.Id AS STRING))) = lower(trim(CAST(rmfal.ForecastId AS STRING)))
               AND rmfal.IsDeleted = 0
            LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimEnterpriseGradeGroup b
                ON lower(trim(
                    CASE WHEN lower(trim(rmfal.RawMaterialName)) = lower(trim('#1 Busheling & Bundles'))
                         THEN 'Busheling'
                         ELSE rmfal.RawMaterialName END
                   )) = lower(trim(b.EnterpriseGradeGroup))
        WHERE ff.IsDeleted = 0
          AND lower(trim(rmfal.Discriminator)) IN (lower(trim('UsageGT')), lower(trim('UsagePercent')))
          AND lower(trim(rmfal.ParentGradeGroupName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.RawMaterialName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.ParentGradeGroupName)) IN (
              lower(trim('Cuts')), lower(trim('Home')), lower(trim('Primes')),
              lower(trim('Secondaries')), lower(trim('Shred')), lower(trim('Subs'))
          )
        GROUP BY
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT),
            mm.Code,
            b.EnterpriseGradeGroupKey,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 4), 'yyyyMMdd') AS INT)

        UNION

        -- Forecast 180 (ForecastPeriod = 6, ConsumptionForecastPeriodDateKey offset = +5 month(s))
        SELECT
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT)  AS ConsumptionForecastDateKey,
            mm.Code                                                                                AS NucorOrgID,
            COALESCE(b.EnterpriseGradeGroupKey, -1)                                               AS EnterpriseGradeGroupKey,
            6                                                                          AS ForecastPeriod,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 5), 'yyyyMMdd') AS INT)  AS ConsumptionForecastPeriodDateKey,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsageGT'))      THEN COALESCE(Forecast180, 0) * 2240.0 ELSE 0 END) AS ForecastWgt,
            SUM(CASE WHEN lower(trim(rmfal.Discriminator)) = lower(trim('UsagePercent')) THEN COALESCE(Forecast90, 0)         ELSE 0 END) AS ForecastPct
        FROM {source_catalog}.djj.ODS_NUE_Forecasts ff
            INNER JOIN {source_catalog}.djj.ODS_MDS_NUEMillNames mm
                ON lower(trim(CAST(ff.DivisionId AS STRING))) = lower(trim(CAST(mm.Code AS STRING)))
               AND mm.IsDeleted = 0
            INNER JOIN {source_catalog}.djj.ODS_NUE_RawMaterialForecastActualLines rmfal
                ON lower(trim(CAST(ff.Id AS STRING))) = lower(trim(CAST(rmfal.ForecastId AS STRING)))
               AND rmfal.IsDeleted = 0
            LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimEnterpriseGradeGroup b
                ON lower(trim(
                    CASE WHEN lower(trim(rmfal.RawMaterialName)) = lower(trim('#1 Busheling & Bundles'))
                         THEN 'Busheling'
                         ELSE rmfal.RawMaterialName END
                   )) = lower(trim(b.EnterpriseGradeGroup))
        WHERE ff.IsDeleted = 0
          AND lower(trim(rmfal.Discriminator)) IN (lower(trim('UsageGT')), lower(trim('UsagePercent')))
          AND lower(trim(rmfal.ParentGradeGroupName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.RawMaterialName)) NOT IN (lower(trim('Unknown')), lower(trim('Not Applicable')))
          AND lower(trim(rmfal.ParentGradeGroupName)) IN (
              lower(trim('Cuts')), lower(trim('Home')), lower(trim('Primes')),
              lower(trim('Secondaries')), lower(trim('Shred')), lower(trim('Subs'))
          )
        GROUP BY
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 0), 'yyyyMMdd') AS INT),
            mm.Code,
            b.EnterpriseGradeGroupKey,
            CAST(DATE_FORMAT(ADD_MONTHS(MAKE_DATE(ff.Year, ff.Month, 1), 5), 'yyyyMMdd') AS INT)

    ) base
    LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimNucorMillLocations nml
        ON lower(trim(CAST(base.NucorOrgID AS STRING))) = lower(trim(CAST(nml.NucorOrgID AS STRING)))
    LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimConsumer cons
        ON lower(trim(CAST(nml.DJJConsumerID AS STRING))) = lower(trim(CAST(cons.ConsumerID AS STRING)))
"""
spark.sql(qs_create_temp)

### Perform Updates

In [ ]:
# Perform Updates
# Original SQL: UPDATE a SET ... FROM [Nue].[factConsumptionForecast] a
#   INNER JOIN #CVPForecastedConsumption b ON matching keys
# Join keys: ConsumptionForecastDateKey, ConsumptionForecastPeriodDateKey,
#            ForecastPeriod, NucorOrgID, EnterpriseGradeGroupKey
# Databricks does not support UPDATE ... FROM; converted to MERGE INTO per Rule 16.
# NUE schema in SQL -> NUE schema in Databricks (star schema layer).
# DJJLastUpdateTime in the star schema target remains DJJLastUpdateTime at write time.

qs_update = f"""
    MERGE INTO {target_catalog}.NUE.factConsumptionForecast a
    USING {target_catalog}.temp.CVPForecastedConsumption b
        ON lower(trim(CAST(a.ConsumptionForecastDateKey AS STRING)))       = lower(trim(CAST(b.ConsumptionForecastDateKey AS STRING)))
       AND lower(trim(CAST(a.ConsumptionForecastPeriodDateKey AS STRING))) = lower(trim(CAST(b.ConsumptionForecastPeriodDateKey AS STRING)))
       AND lower(trim(CAST(a.ForecastPeriod AS STRING)))                   = lower(trim(CAST(b.ForecastPeriod AS STRING)))
       AND lower(trim(CAST(a.NucorOrgID AS STRING)))                       = lower(trim(CAST(b.NucorOrgID AS STRING)))
       AND lower(trim(CAST(a.EnterpriseGradeGroupKey AS STRING)))          = lower(trim(CAST(b.EnterpriseGradeGroupKey AS STRING)))
    WHEN MATCHED THEN UPDATE SET
        a.ConsumerKey             = b.ConsumerKey,
        a.EnterpriseGradeGroupKey = b.EnterpriseGradeGroupKey,
        a.ForecastWgt             = b.ForecastWgt,
        a.ForecastPct             = b.ForecastPct,
        a.DJJGeneratedFlag        = b.DJJGeneratedFlag,
        a.ETLSSExecutionID        = b.ETLSSExecutionID,
        a.DJJLastUpdateTime       = b.EnrichedTimestampUTC
"""
spark.sql(qs_update)

# Capture rows updated by counting matched rows between destination and staging
RowsUpdated = spark.sql(f"""
    SELECT COUNT(1) AS cnt
    FROM {federated_starschema_catalog}.NUE.factConsumptionForecast a
    INNER JOIN {target_catalog}.temp.CVPForecastedConsumption b
        ON lower(trim(CAST(a.ConsumptionForecastDateKey AS STRING)))       = lower(trim(CAST(b.ConsumptionForecastDateKey AS STRING)))
       AND lower(trim(CAST(a.ConsumptionForecastPeriodDateKey AS STRING))) = lower(trim(CAST(b.ConsumptionForecastPeriodDateKey AS STRING)))
       AND lower(trim(CAST(a.ForecastPeriod AS STRING)))                   = lower(trim(CAST(b.ForecastPeriod AS STRING)))
       AND lower(trim(CAST(a.NucorOrgID AS STRING)))                       = lower(trim(CAST(b.NucorOrgID AS STRING)))
       AND lower(trim(CAST(a.EnterpriseGradeGroupKey AS STRING)))          = lower(trim(CAST(b.EnterpriseGradeGroupKey AS STRING)))
""").first()["cnt"]
print(f"Rows updated: {RowsUpdated}")

### Perform Inserts (new rows only)

In [ ]:
# Perform Inserts
# Original SQL: INSERT INTO [Nue].[factConsumptionForecast]
#   SELECT ... FROM #CVPForecastedConsumption a
#     LEFT OUTER JOIN [Nue].[factConsumptionForecast] b ON matching keys
#   WHERE b.ConsumptionForecastDateKey IS NULL
# Converted to MERGE INTO WHEN NOT MATCHED INSERT per Rule 16.

qs_insert = f"""
    MERGE INTO {target_catalog}.NUE.factConsumptionForecast a
    USING {target_catalog}.temp.CVPForecastedConsumption b
        ON lower(trim(CAST(a.ConsumptionForecastDateKey AS STRING)))       = lower(trim(CAST(b.ConsumptionForecastDateKey AS STRING)))
       AND lower(trim(CAST(a.ConsumptionForecastPeriodDateKey AS STRING))) = lower(trim(CAST(b.ConsumptionForecastPeriodDateKey AS STRING)))
       AND lower(trim(CAST(a.ForecastPeriod AS STRING)))                   = lower(trim(CAST(b.ForecastPeriod AS STRING)))
       AND lower(trim(CAST(a.NucorOrgID AS STRING)))                       = lower(trim(CAST(b.NucorOrgID AS STRING)))
       AND lower(trim(CAST(a.EnterpriseGradeGroupKey AS STRING)))          = lower(trim(CAST(b.EnterpriseGradeGroupKey AS STRING)))
    WHEN NOT MATCHED THEN INSERT (
        ConsumptionForecastDateKey,
        NucorOrgID,
        EnterpriseGradeGroupKey,
        ConsumerKey,
        ForecastPeriod,
        ConsumptionForecastPeriodDateKey,
        ForecastWgt,
        ForecastPct,
        DJJGeneratedFlag,
        ETLSSExecutionID,
        DJJLastUpdateTime
    ) VALUES (
        b.ConsumptionForecastDateKey,
        b.NucorOrgID,
        b.EnterpriseGradeGroupKey,
        b.ConsumerKey,
        b.ForecastPeriod,
        b.ConsumptionForecastPeriodDateKey,
        b.ForecastWgt,
        b.ForecastPct,
        b.DJJGeneratedFlag,
        b.ETLSSExecutionID,
        b.EnrichedTimestampUTC
    )
"""
spark.sql(qs_insert)

# Capture rows inserted: count staging rows with no matching destination row
RowsInserted = spark.sql(f"""
    SELECT COUNT(1) AS cnt
    FROM {target_catalog}.temp.CVPForecastedConsumption b
    LEFT OUTER JOIN {federated_starschema_catalog}.NUE.factConsumptionForecast a
        ON lower(trim(CAST(b.ConsumptionForecastDateKey AS STRING)))       = lower(trim(CAST(a.ConsumptionForecastDateKey AS STRING)))
       AND lower(trim(CAST(b.ConsumptionForecastPeriodDateKey AS STRING))) = lower(trim(CAST(a.ConsumptionForecastPeriodDateKey AS STRING)))
       AND lower(trim(CAST(b.ForecastPeriod AS STRING)))                   = lower(trim(CAST(a.ForecastPeriod AS STRING)))
       AND lower(trim(CAST(b.NucorOrgID AS STRING)))                       = lower(trim(CAST(a.NucorOrgID AS STRING)))
       AND lower(trim(CAST(b.EnterpriseGradeGroupKey AS STRING)))          = lower(trim(CAST(a.EnterpriseGradeGroupKey AS STRING)))
    WHERE a.ConsumptionForecastDateKey IS NULL
""").first()["cnt"]
print(f"Rows inserted: {RowsInserted}")

### Perform Deletes (grade mapping cleanup – destination rows absent from source)

In [ ]:
# Perform Deletes
# Original SQL:
#   DELETE a FROM [Nue].[factConsumptionForecast] a
#     LEFT OUTER JOIN #CVPForecastedConsumption b ON matching keys
#   WHERE b.ConsumptionForecastDateKey IS NULL
# Purpose: remove destination rows that no longer exist in the source
#   (handles grade mapping updates that change EnterpriseGradeGroupKey).
# Original: SELECT @RowsDeleted = @RowsDeleted (no new count assigned);
#   RowsDeleted captured as count of rows identified for deletion here.
# Databricks does not support DELETE with JOIN (Rule 15):
#   Step 1 – identify rows to delete via TEMP VIEW
#   Step 2 – DELETE using composite key IN subquery

# Step 1: Identify destination rows not present in the staging data
spark.sql(f"""CREATE OR REPLACE TEMP VIEW vw_factConsumptionForecast_to_delete AS
    SELECT
        a.ConsumptionForecastDateKey,
        a.ConsumptionForecastPeriodDateKey,
        a.ForecastPeriod,
        a.NucorOrgID,
        a.EnterpriseGradeGroupKey
    FROM {federated_starschema_catalog}.NUE.factConsumptionForecast a
        LEFT OUTER JOIN {target_catalog}.temp.CVPForecastedConsumption b
            ON lower(trim(CAST(a.ConsumptionForecastDateKey AS STRING)))       = lower(trim(CAST(b.ConsumptionForecastDateKey AS STRING)))
           AND lower(trim(CAST(a.ConsumptionForecastPeriodDateKey AS STRING))) = lower(trim(CAST(b.ConsumptionForecastPeriodDateKey AS STRING)))
           AND lower(trim(CAST(a.ForecastPeriod AS STRING)))                   = lower(trim(CAST(b.ForecastPeriod AS STRING)))
           AND lower(trim(CAST(a.NucorOrgID AS STRING)))                       = lower(trim(CAST(b.NucorOrgID AS STRING)))
           AND lower(trim(CAST(a.EnterpriseGradeGroupKey AS STRING)))          = lower(trim(CAST(b.EnterpriseGradeGroupKey AS STRING)))
    WHERE b.ConsumptionForecastDateKey IS NULL
""")

# Capture rows deleted before executing delete
RowsDeleted = spark.sql(
    "SELECT COUNT(1) AS cnt FROM vw_factConsumptionForecast_to_delete"
).first()["cnt"]

# Step 2: Delete identified rows from the destination fact table
spark.sql(f"""
    DELETE FROM {target_catalog}.NUE.factConsumptionForecast
    WHERE (
        ConsumptionForecastDateKey,
        ConsumptionForecastPeriodDateKey,
        ForecastPeriod,
        NucorOrgID,
        EnterpriseGradeGroupKey
    ) IN (
        SELECT
            ConsumptionForecastDateKey,
            ConsumptionForecastPeriodDateKey,
            ForecastPeriod,
            NucorOrgID,
            EnterpriseGradeGroupKey
        FROM vw_factConsumptionForecast_to_delete
    )
""")

print(f"Rows deleted: {RowsDeleted}")

In [ ]:
# Get final row count of the target fact table after all ETL operations
TableFinalRowCount = spark.sql(f"""
    SELECT COUNT(1) AS cnt
    FROM {target_catalog}.NUE.factConsumptionForecast
""").first()["cnt"]
print(f"Final row count: {TableFinalRowCount}")

In [ ]:
# ETL Execution Log - close out with final counts and success flag
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success='1',
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [ ]:
# Set ETL Master IncrementalExtractTime, ETLTemplate, ETLType
# Original: UPDATE DJJMetadata.dbo.ETLMaster SET ... WHERE ETLID = @ETLID
# Databricks does not support UPDATE ... FROM; using MERGE INTO per Rule 16.

qs_etl_master = f"""
    MERGE INTO {target_catalog}.metadata.ETLMaster tgt
    USING (
        SELECT
            '{IncrementalExtractTime}'  AS IncrementalExtractTime,
            '{ETLTemplate}'             AS ETLTemplate,
            '{ETLType}'                 AS ETLType,
            {ETLID}                     AS ETLID
    ) src
        ON lower(trim(CAST(tgt.ETLID AS STRING))) = lower(trim(CAST(src.ETLID AS STRING)))
    WHEN MATCHED THEN UPDATE SET
        tgt.IncrementalExtractTime = src.IncrementalExtractTime,
        tgt.ETLTemplate            = src.ETLTemplate,
        tgt.ETLType                = src.ETLType
"""
spark.sql(qs_etl_master)

In [ ]:
# Cleanup - Drop the temp Delta table created during this ETL run
spark.sql(f"DROP TABLE IF EXISTS {target_catalog}.temp.CVPForecastedConsumption")
print("Temp table dropped. ETL complete.")